In [ ]:
import requests
import pandas as pd
from datetime import datetime, timezone

BASE = "https://api.binance.com"
SYMBOL = "BTCUSDT"

# --- 1. Current price (ticker) ---
ticker = requests.get(f"{BASE}/api/v3/ticker/price", params={"symbol": SYMBOL}).json()
print(f"Price: {ticker['price']:} USDT")

# --- 2. 24h stats ---
stats = requests.get(f"{BASE}/api/v3/ticker/24hr", params={"symbol": SYMBOL}).json()
stats_df = pd.DataFrame([{
    "symbol":        stats["symbol"],
    "last_price":    float(stats["lastPrice"]),
    "open":          float(stats["openPrice"]),
    "high":          float(stats["highPrice"]),
    "low":           float(stats["lowPrice"]),
    "volume_btc":    float(stats["volume"]),
    "volume_usdt":   float(stats["quoteVolume"]),
    "price_change%": float(stats["priceChangePercent"]),
    "trades":        stats["count"],
    "timestamp":     datetime.fromtimestamp(stats["closeTime"] / 1000, tz=timezone.utc),
}])
print("\n24h Stats:")
display(stats_df.T)

# --- 3. Order book snapshot (top 20 levels) ---
depth = requests.get(f"{BASE}/api/v3/depth", params={"symbol": SYMBOL, "limit": 20}).json()

bids = pd.DataFrame(depth["bids"], columns=["price", "qty"], dtype=float)
asks = pd.DataFrame(depth["asks"], columns=["price", "qty"], dtype=float)

bids["side"] = "bid"
asks["side"] = "ask"
book = pd.concat([bids, asks], ignore_index=True)

print(f"\nOrder book snapshot  (last update id: {depth['lastUpdateId']})")
print("\nTop 5 bids:")
display(bids.head())
print("\nTop 5 asks:")
display(asks.head())

# --- 4. Recent trades (last 20) ---
trades_raw = requests.get(f"{BASE}/api/v3/trades", params={"symbol": SYMBOL, "limit": 20}).json()
trades_df = pd.DataFrame(trades_raw)[["id", "price", "qty", "time", "isBuyerMaker"]]
trades_df[["price", "qty"]] = trades_df[["price", "qty"]].astype(float)
trades_df["time"] = pd.to_datetime(trades_df["time"], unit="ms", utc=True)
trades_df.rename(columns={"isBuyerMaker": "sell"}, inplace=True)

print("\nRecent trades:")
display(trades_df)

Price: 80913.59000000 USDT

24h Stats:


,0
symbol,BTCUSDT
last_price,80913.59
open,80092.64
high,81791.48
low,80040.0
volume_btc,17195.94748
volume_usdt,1394872254.604011
price_change%,1.025
trades,2513150
timestamp,2026-05-06 00:45:06+00:00



Order book snapshot  (last update id: 93225173694)

Top 5 bids:


,price,qty,side
0,80913.58,2.39475,bid
1,80913.57,0.00042,bid
2,80913.11,0.00007,bid
3,80912.34,0.00014,bid
4,80912.33,0.01002,bid



Top 5 asks:


,price,qty,side
0,80913.59,1.41247,ask
1,80913.60,0.00063,ask
2,80914.00,0.00024,ask
3,80914.77,0.00007,ask
4,80914.94,0.00014,ask



Recent trades:


,id,price,qty,time,sell
0,6275696014,80913.11,0.00007,2026-05-06 00:45:01.567000+00:00,False
1,6275696015,80913.11,0.00007,2026-05-06 00:45:01.567000+00:00,False
2,6275696016,80913.11,0.00007,2026-05-06 00:45:01.567000+00:00,False
3,6275696017,80913.12,0.00007,2026-05-06 00:45:01.567000+00:00,False
4,6275696018,80913.12,0.00042,2026-05-06 00:45:01.567000+00:00,False
5,6275696019,80913.12,0.03958,2026-05-06 00:45:01.569000+00:00,False
6,6275696020,80913.59,0.00430,2026-05-06 00:45:01.570000+00:00,False
7,6275696021,80913.59,0.00737,2026-05-06 00:45:01.570000+00:00,False
8,6275696022,80913.58,0.00024,2026-05-06 00:45:01.730000+00:00,True
9,6275696023,80913.59,0.00007,2026-05-06 00:45:01.783000+00:00,False


In [ ]:
import requests
import pandas as pd
from datetime import datetime, timezone

# Futures (USD-M perpetual) base URL — no API key required
FBASE  = "https://fapi.binance.com"
SYMBOL = "BTCUSDT"

# --- 1. Current mark / index / last price ---
premium = requests.get(f"{FBASE}/fapi/v1/premiumIndex", params={"symbol": SYMBOL}).json()
print(f"Last price:   {premium['lastFundingRate']}  (funding rate)")
print(f"Mark price:   {premium['markPrice']}")
print(f"Index price:  {premium['indexPrice']}")
print(f"Next funding: {datetime.fromtimestamp(premium['nextFundingTime']/1000, tz=timezone.utc)}")

# --- 2. 24h stats ---
stats = requests.get(f"{FBASE}/fapi/v1/ticker/24hr", params={"symbol": SYMBOL}).json()
stats_df = pd.DataFrame([{
    "symbol":        stats["symbol"],
    "last_price":    float(stats["lastPrice"]),
    "open":          float(stats["openPrice"]),
    "high":          float(stats["highPrice"]),
    "low":           float(stats["lowPrice"]),
    "volume_btc":    float(stats["volume"]),
    "volume_usdt":   float(stats["quoteVolume"]),
    "price_change%": float(stats["priceChangePercent"]),
    "trades":        stats["count"],
    "timestamp":     datetime.fromtimestamp(stats["closeTime"] / 1000, tz=timezone.utc),
}])
print("\n24h Stats (perp):")
display(stats_df.T)

# --- 3. Order book snapshot (top 20 levels) ---
depth = requests.get(f"{FBASE}/fapi/v1/depth", params={"symbol": SYMBOL, "limit": 20}).json()

bids = pd.DataFrame(depth["bids"], columns=["price", "qty"], dtype=float)
asks = pd.DataFrame(depth["asks"], columns=["price", "qty"], dtype=float)
bids["side"] = "bid"
asks["side"] = "ask"
book_perp = pd.concat([bids, asks], ignore_index=True)

print(f"\nOrder book snapshot  (last update id: {depth['lastUpdateId']})")
print("\nTop 5 bids:")
display(bids.head())
print("\nTop 5 asks:")
display(asks.head())

# --- 4. Recent trades (last 20) ---
trades_raw = requests.get(f"{FBASE}/fapi/v1/trades", params={"symbol": SYMBOL, "limit": 20}).json()
trades_df = pd.DataFrame(trades_raw)[["id", "price", "qty", "time", "isBuyerMaker"]]
trades_df[["price", "qty"]] = trades_df[["price", "qty"]].astype(float)
trades_df["time"] = pd.to_datetime(trades_df["time"], unit="ms", utc=True)
trades_df.rename(columns={"isBuyerMaker": "sell"}, inplace=True)

print("\nRecent trades (perp):")
display(trades_df)

# --- 5. Open interest ---
oi = requests.get(f"{FBASE}/fapi/v1/openInterest", params={"symbol": SYMBOL}).json()
print(f"\nOpen interest: {float(oi['openInterest']):.2f} BTC  "
      f"(as of {datetime.fromtimestamp(oi['time']/1000, tz=timezone.utc)})")

Last price:   -0.00009116  (funding rate)
Mark price:   80943.90000000
Index price:  80962.70847826
Next funding: 2026-05-06 08:00:00+00:00

24h Stats (perp):


,0
symbol,BTCUSDT
last_price,80933.4
open,80050.0
high,81745.4
low,80031.5
volume_btc,174388.349
volume_usdt,14143039737.059999
price_change%,1.104
trades,3584354
timestamp,2026-05-06 00:51:34.515000+00:00



Order book snapshot  (last update id: 10480700182857)

Top 5 bids:


,price,qty,side
0,80943.9,6.386,bid
1,80943.8,0.157,bid
2,80943.7,0.002,bid
3,80943.6,0.002,bid
4,80943.5,0.006,bid



Top 5 asks:


,price,qty,side
0,80944.0,5.217,ask
1,80944.1,0.015,ask
2,80944.2,0.001,ask
3,80944.4,0.004,ask
4,80944.5,0.002,ask



Recent trades (perp):


,id,price,qty,time,sell
0,7633412579,80944.0,0.049,2026-05-06 00:51:41.354000+00:00,False
1,7633412580,80944.0,0.322,2026-05-06 00:51:41.354000+00:00,False
2,7633412581,80944.0,0.002,2026-05-06 00:51:41.354000+00:00,False
3,7633412582,80944.0,0.001,2026-05-06 00:51:41.354000+00:00,False
4,7633412583,80944.0,0.263,2026-05-06 00:51:41.354000+00:00,False
5,7633412584,80944.0,0.029,2026-05-06 00:51:41.354000+00:00,False
6,7633412585,80944.0,0.065,2026-05-06 00:51:41.354000+00:00,False
7,7633412586,80944.0,0.282,2026-05-06 00:51:41.432000+00:00,False
8,7633412587,80944.0,0.520,2026-05-06 00:51:41.432000+00:00,False
9,7633412588,80944.0,0.006,2026-05-06 00:51:41.432000+00:00,False



Open interest: 113746.86 BTC  (as of 2026-05-06 00:51:36.479000+00:00)
